In [2]:
%load_ext autoreload
%autoreload 2

In [ ]:
# https://github.com/pymupdf/pymupdf
import pymupdf

doc = pymupdf.open("../data/raw/resumes/data-ai/AI_Engineer_candidate_ai-engineer-data-engineer-9868273.pdf")
for page in doc:
    data = page.get_text()
    print(data)

In [4]:
page = doc[0]

blocks = page.get_text("dict")["blocks"]
for block in blocks:
    if block["type"] == 0:  # text block
        for line in block["lines"]:
            for span in line["spans"]:
                print(f"{span['text']!r}  font={span['font']}  size={span['size']:.1f}")

'Viet Nguyen'  font=Calibri-Bold  size=24.0
'vietnguyen1602.github.io/vietnguyen1602'  font=Calibri-Italic  size=12.0
'github.com/vietnguyen1602'  font=Calibri-Italic  size=10.0
'Experience'  font=Calibri  size=14.0
'December 2021'  font=Calibri  size=10.0
'Viettel Software Service, '  font=Calibri-Bold  size=10.0
'AI Intern'  font=Calibri-Italic  size=10.0
'\uf0a7'  font=Wingdings-Regular  size=10.0
'Develop smart office system for VSS and Viettel Store'  font=Calibri  size=10.0
'\uf0a7'  font=Wingdings-Regular  size=10.0
'Building a face attendance system (with and without masks)'  font=Calibri  size=10.0
'\uf0a7'  font=Wingdings-Regular  size=10.0
'Develop chatbot'  font=Calibri  size=10.0
'\uf0a7'  font=Wingdings-Regular  size=10.0
'Technologies: '  font=Calibri-Bold  size=10.0
'Tensorflow, InsightFace, NLTK'  font=Calibri  size=10.0
'April 2022'  font=Calibri  size=10.0
'Topica Edtech Group – Study Now, '  font=Calibri-Bold  size=10.0
'AI Engineer – Data Engineer'  font=Calibri-It

In [5]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

# Load model and tokenizer
model_name = "yashpwr/resume-ner-bert-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

# Example resume text
# text = "John Smith is a senior software engineer with 8 years of experience at Google. He has expertise in Python, JavaScript, and machine learning. Contact: john.smith@gmail.com"

# Tokenize
inputs = tokenizer(
    data,
    return_tensors="pt",
    truncation=True,
    max_length=2048,
    padding=True
)

# Predict
with torch.no_grad():
    outputs = model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=2)

# Extract entities
entities = []
current_entity = None

for i, pred in enumerate(predictions[0]):
    label = model.config.id2label[pred.item()]
    token_id = inputs["input_ids"][0][i].item()
    token = tokenizer.convert_ids_to_tokens(token_id)
    
    if label.startswith('B-'):
        if current_entity:
            entities.append(current_entity)
        current_entity = {
            'text': token,
            'label': label[2:],  # Remove 'B-' prefix
            'start': i
        }
    elif label.startswith('I-') and current_entity:
        current_entity['text'] += ' ' + token
    elif label == 'O':
        if current_entity:
            entities.append(current_entity)
            current_entity = None

if current_entity:
    entities.append(current_entity)

print("Extracted Entities:")
for entity in entities:
    print(f"- {entity['label']}: {entity['text']}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Extracted Entities:
- Skills: Skills
- Degree: Machine
- Companies worked at: and without masks , V ##SS Dev ##elo ##p the attendance system


In [ ]:
import json
from transformers import AutoModelForCausalLM, AutoTokenizer


def predict_NuExtract(model, tokenizer, text, schema, example=["","",""]):
    schema = json.dumps(json.loads(schema), indent=4)
    input_llm =  "<|input|>\n### Template:\n" +  schema + "\n"
    for i in example:
      if i != "":
          input_llm += "### Example:\n"+ json.dumps(json.loads(i), indent=4)+"\n"
    
    input_llm +=  "### Text:\n"+text +"\n<|output|>\n"
    input_ids = tokenizer(input_llm, return_tensors="pt", truncation=True, max_length=4000).to("cuda")

    output = tokenizer.decode(model.generate(**input_ids)[0], skip_special_tokens=True)
    return output.split("<|output|>")[1].split("<|end-output|>")[0]


model = AutoModelForCausalLM.from_pretrained("numind/NuExtract-tiny", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("numind/NuExtract-tiny", trust_remote_code=True)

model.to("cuda")

model.eval()

text = """We introduce Mistral 7B, a 7–billion-parameter language model engineered for
superior performance and efficiency. Mistral 7B outperforms the best open 13B
model (Llama 2) across all evaluated benchmarks, and the best released 34B
model (Llama 1) in reasoning, mathematics, and code generation. Our model
leverages grouped-query attention (GQA) for faster inference, coupled with sliding
window attention (SWA) to effectively handle sequences of arbitrary length with a
reduced inference cost. We also provide a model fine-tuned to follow instructions,
Mistral 7B – Instruct, that surpasses Llama 2 13B – chat model both on human and
automated benchmarks. Our models are released under the Apache 2.0 license.
Code: https://github.com/mistralai/mistral-src
Webpage: https://mistral.ai/news/announcing-mistral-7b/"""

schema = """{
    "Model": {
        "Name": "",
        "Number of parameters": "",
        "Number of max token": "",
        "Architecture": []
    },
    "Usage": {
        "Use case": [],
        "Licence": ""
    }
}"""

prediction = predict_NuExtract(model, tokenizer, text, schema, example=["","",""])
print(prediction)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx